In [2]:
!python -m pip install libigl

In [3]:
import os, sys

from math import ceil
import pyvista as pv
import numpy as np
import meshplot as mp
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, FloatSlider
from skimage import measure
from scipy.ndimage import zoom
from scipy.interpolate import interpn
from IPython.display import display
from einops import rearrange
import igl
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import torch
from scipy import stats
import matplotlib.pyplot as plt
import pandas as pd
import open3d as o3d
from IPython.display import display
import random

In [4]:
# Meshplot left an annoying print statement in their code. Using this context manager to supress it...
class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

In [9]:
# Dot product on the first dimension of n-dimensional arrays x and y
def dot(x, y):
    return np.einsum('i..., i... -> ...', x, y)

# Signed distance functions from Inigo Quilez https://iquilezles.org/articles/distfunctions/
# You could implement the smooth minimum operation as well to compose shapes together for more complex situations
def sdf_sphere(x, radius):
    return np.linalg.norm(x, axis=0) - radius

def sdf_capsule(x, a, b, r):
    xa = coords - a
    ba = coords - a
    h = np.clip(dot(xa, ba) / dot(ba, ba), 0., 1.)
    return np.linalg.norm(xa - ba * h) - r

def sdf_torus(x, radius, thickness):
    
    q = np.stack([np.linalg.norm(x[[0, 1]], axis=0) - radius, x[2]])
    return np.linalg.norm(q, axis=0) - thickness

# Crop an n-dimensional image with a centered cropping region
def center_crop(img, shape):
    start = [a // 2 - da // 2 for a, da in zip(img.shape, shape)]
    end = [a + b for a, b in zip(start, shape)]
    slices = tuple([slice(a, b) for a, b in zip(start, end)])
    return img[slices]

# Add noise to coordinates
def gradient_noise(x, scale, strength, seed=None):
    shape = [ceil(s / scale) for s in x.shape[1:]]
    if seed:
        np.random.seed(seed)
    scalar_noise = np.random.randn(*shape)
    scalar_noise = zoom(scalar_noise, zoom=scale)
    scalar_noise = center_crop(scalar_noise, shape=x.shape[1:])
    vector_noise = np.stack(np.gradient(scalar_noise))
    return vector_noise * strength


In [9]:
plot=None
@mp.interact(
    radius=(0, 0.3, 0.01), 
    thickness=(0.01, 0.1, 0.01), 
    noise_scale=(5, 25), 
    noise_strength=(0.0, 0.4, 0.05),
    seed=(1, 100)
)
def show(radius, thickness, noise_scale, noise_strength, seed):
    global plot
    global sdf
    coords = np.linspace(-1, 1, 100)
    x = np.stack(np.meshgrid(coords, coords, coords))
    x = x + gradient_noise(x, noise_scale, noise_strength, seed)
    sdf = sdf_torus(x, radius, thickness)
    verts, faces, normals, values = measure.marching_cubes(sdf, level=0)
    
    if plot is None:
        plot = mp.plot(verts, faces, return_plot=True)
    else:
        with HiddenPrints():
            plot.update_object(vertices=verts, faces=faces)
        display(plot._renderer)

interactive(children=(FloatSlider(value=0.15, description='radius', max=0.3, step=0.01), FloatSlider(value=0.0…

In [11]:
plot=None
@mp.interact(
    radius=(0, 0.5, 0.01), 
    thickness=(0.01, 0.20, 0.01), 
    noise_scale=(0.0, 20, 1),
    noise_strength=(0.0, 10, 1),
    seed=(1, 100),
    bump_angle=(-1., 1., 0.01),
    bump_width=(0.01, 0.02, 0.001),
    bump_height=(0.01, 50.),
    scale=(0.85, 1.15, 0.01),
)
def show(radius, thickness, noise_scale, noise_strength, seed, bump_angle, bump_width, bump_height, scale):
    global plot
    coords = np.linspace(-1, 1, 100)
    x = np.stack(np.meshgrid(coords, coords, coords))
    sdf = sdf_torus(x, radius, thickness)
    verts, faces, normals, values = measure.marching_cubes(sdf, level=0)
    
    x_warp = gradient_noise(x, noise_scale, noise_strength, seed)
    print(x_warp.shape)
    
    angle = np.pi * bump_angle
    gaussian_center = np.array([np.cos(angle), np.sin(angle), 0.]) * radius
    print(gaussian_center)
    x_dist = np.linalg.norm((x - gaussian_center[:, None, None, None]), axis=0)
    print(x_dist.shape)
    x_bump = bump_height * np.e ** -(1. / bump_width * x_dist ** 2)
    print(x_bump.shape)
    x_warp += -np.stack(np.gradient(x_bump))
    
    x_warp = rearrange(x_warp, 'v h w d -> h w d v')
    vertex_noise = interpn([np.arange(100) for _ in range(3)], x_warp, verts)
    verts += vertex_noise
    
    original_center = np.mean(verts, axis=0)
    
    verts = verts*scale
    
    new_center = np.mean(verts, axis=0)
    displacement_vector = original_center - new_center
    verts += displacement_vector
    
    print(verts.shape)
    print(faces.shape)
    max_values = np.amax(verts, axis=0)
    min_values = np.amin(verts, axis=0)

    print("Maximum values:", max_values)
    print("Minimum values:", min_values)
    if plot is None:
        plot = mp.plot(verts, faces, return_plot=True)
    else:
        with HiddenPrints():
            plot.update_object(vertices=verts, faces=faces)
        display(plot._renderer)

interactive(children=(FloatSlider(value=0.25, description='radius', max=0.5, step=0.01), FloatSlider(value=0.0…

## Save torus data into file

In [10]:
from pathlib import Path

In [19]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from skimage import measure
from scipy.interpolate import interpn
from einops import rearrange

out_dir = "torus_bump_1000_angle_classification"
os.makedirs(out_dir, exist_ok=True)

TOTAL = 1000
PER_CLASS = 500
grid_N = 100

radius = 0.25
thickness = 0.10

# FIXED (no variability besides is_bump & angle)
scale_fixed = 1.0
bump_height_fixed = 30.0
bump_width_fixed = 0.01

# Fixed background noise pattern
noise_scale = 16
noise_strength = 10
seed = 50

# Angle per class: 0..1 (normalized). Geometry uses 2*pi*angle01 when is_bump=1.
angles01_class = np.linspace(0.0, 1.0, PER_CLASS, endpoint=False)

# For label normalization of s & b (constants)
feature_range_scale = np.linspace(0.85, 1.15, 300, endpoint=False)
feature_range_bump_height = np.linspace(20, 45, 300, endpoint=False)
s_const = MinMaxScaler().fit(feature_range_scale.reshape(-1,1)).transform([[scale_fixed]]).item()
b_const = MinMaxScaler().fit(feature_range_bump_height.reshape(-1,1)).transform([[bump_height_fixed]]).item()

# Grid & base SDF (reused)
coords = np.linspace(-1, 1, grid_N)
x = np.stack(np.meshgrid(coords, coords, coords))
base_sdf = sdf_torus(x, radius, thickness)

# Fixed background field
x_warp_bg = gradient_noise(x, noise_scale, noise_strength, seed=seed)

labels = {}
rows = []
idx_global = 0

def make_sample(is_bump, angle01, idx_global):
    # Base mesh
    verts, faces, normals, values = measure.marching_cubes(base_sdf, level=0)

    # Warp = fixed background + optional bump
    x_warp = x_warp_bg.copy()
    if is_bump == 1:
        angle_rad = 2.0 * np.pi * float(angle01)  # [0, 2π)
        center = np.array([np.cos(angle_rad), np.sin(angle_rad), 0.0]) * radius
        x_dist = np.linalg.norm((x - center[:, None, None, None]), axis=0)
        x_bump = bump_height_fixed * np.e ** (-(1.0 / bump_width_fixed) * (x_dist ** 2))
        x_warp += -np.stack(np.gradient(x_bump))
    # sample and apply warp
    x_warp = rearrange(x_warp, 'v h w d -> h w d v')
    vertex_noise = interpn([np.arange(grid_N) for _ in range(3)], x_warp, verts, bounds_error=False, fill_value=0.0)
    verts = verts + vertex_noise

    # scale (fixed) and re-center
    c0 = np.mean(verts, axis=0)
    verts = verts * scale_fixed
    c1 = np.mean(verts, axis=0)
    verts += (c0 - c1)

    # save
    fname = f"torus_bump_{idx_global:04d}"
    fpath = os.path.join(out_dir, f"{fname}.ply")
    write_mesh(fpath, verts, faces)

    # labels: [is_bump, angle01 (0..1), s_const, b_const]
    labels[fname] = np.array([is_bump, float(angle01), s_const, b_const], dtype=np.float32)
    rows.append({
        "shape": fname,
        "is_bump": is_bump,
        "angle": float(angle01),
        "scale": s_const,
        "bump_height": b_const
    })

# --- Generate class 1 (bump present) ---
for a01 in tqdm(angles01_class, desc="Class 1 (bump)"):
    make_sample(is_bump=1, angle01=a01, idx_global=idx_global)
    idx_global += 1

# --- Generate class 0 (no bump) ---
for a01 in tqdm(angles01_class, desc="Class 0 (no bump)"):
    make_sample(is_bump=0, angle01=a01, idx_global=idx_global)
    idx_global += 1

# Save labels  (ONLY CHANGE: convert np.array -> torch.Tensor before saving)
safe_labels = {
    k: (torch.as_tensor(v, dtype=torch.float32) if isinstance(v, np.ndarray) else v)
    for k, v in labels.items()
}
torch.save(safe_labels, os.path.join(out_dir, "labels.pt"))
pd.DataFrame(rows).to_csv(os.path.join(out_dir, "torus_bump_labels.csv"), index=False)

print(f"Done. Saved {TOTAL} meshes + labels to: {out_dir}")


Class 0 (no bump): 100%|██████████| 500/500 [00:06<00:00, 78.09it/s]

Done. Saved 1000 meshes + labels to: torus_bump_1000_angle_classification


In [16]:
# -- optional writer fallback (igl -> open3d) --
try:
    import igl
    def write_mesh(path, V, F):
        igl.write_triangle_mesh(path, V.astype(np.float64), F.astype(np.int32))
except Exception:
    import open3d as o3d
    def write_mesh(path, V, F):
        tri = o3d.geometry.TriangleMesh()
        tri.vertices = o3d.utility.Vector3dVector(V.astype(np.float64))
        tri.triangles = o3d.utility.Vector3iVector(F.astype(np.int32))
        o3d.io.write_triangle_mesh(path, tri)

In [20]:
import os, glob
import numpy as np
import imageio
from PIL import Image, ImageDraw, ImageFont
import torch
import pyvista as pv

# --- Config ---
data_dir = "torus_bump_1000_angle_classification"   # change if needed
out_mp4  = os.path.join(data_dir, "dataset_traverse.mp4")
fps      = 24
width, height = 800, 600
bg_color = "black"
mesh_color = "lightblue"

# --- Load files & labels ---
ply_files = sorted(p for p in glob.glob(os.path.join(data_dir, "*.ply"))
                   if os.path.basename(p).startswith("torus_bump_"))
if not ply_files:
    raise FileNotFoundError(f"No .ply found in {data_dir}")

labels = torch.load(os.path.join(data_dir, "labels.pt"), map_location="cpu", weights_only=False)

# --- Helpers ---
def label_lines(name):
    arr = labels[name]
    return [
        f"{name}",
        f"is_bump: {int(arr[0])}",
        f"angle01: {arr[1]:.4f}",
        f"scale_norm: {arr[2]:.4f}",
        f"bump_h_norm: {arr[3]:.4f}",
    ]

def overlay_text(frame_rgb, text_lines, xy=(10, 10), color=(255,255,255)):
    img = Image.fromarray(frame_rgb)
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default()
    except:
        font = None
    x, y = xy
    for line in text_lines:
        draw.text((x, y), line, fill=color, font=font)
        y += 18
    return np.array(img)

# --- Setup plotter ---
pv.start_xvfb()  # safe on Linux/mac; harmless otherwise
plotter = pv.Plotter(off_screen=True, window_size=(width, height))
plotter.set_background(bg_color)

# --- Write video ---
writer = imageio.get_writer(out_mp4, fps=fps, codec="libx264", quality=8)
print(f"[INFO] Writing video: {out_mp4}")

for ply in ply_files:
    name = os.path.splitext(os.path.basename(ply))[0]
    print(f"{name}: {labels[name].tolist()}")

    plotter.clear()
    mesh = pv.read(ply)
    mesh = mesh.compute_normals(auto_orient_normals=True)
    plotter.add_mesh(mesh, color=mesh_color, smooth_shading=True)

    plotter.camera_position = "iso"
    frame = plotter.screenshot(return_img=True)
    frame = overlay_text(frame, label_lines(name))
    writer.append_data(frame)

writer.close()
plotter.close()
print("[INFO] Done! Saved video to:", out_mp4)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (800, 600) to (800, 608) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


[INFO] Writing video: torus_bump_1000_angle_classification/dataset_traverse.mp4
torus_bump_0000: [1.0, 0.0, 0.5016722679138184, 0.40133780241012573]
torus_bump_0001: [1.0, 0.0020000000949949026, 0.5016722679138184, 0.40133780241012573]
torus_bump_0002: [1.0, 0.004000000189989805, 0.5016722679138184, 0.40133780241012573]
torus_bump_0003: [1.0, 0.006000000052154064, 0.5016722679138184, 0.40133780241012573]
torus_bump_0004: [1.0, 0.00800000037997961, 0.5016722679138184, 0.40133780241012573]
torus_bump_0005: [1.0, 0.009999999776482582, 0.5016722679138184, 0.40133780241012573]
torus_bump_0006: [1.0, 0.012000000104308128, 0.5016722679138184, 0.40133780241012573]
torus_bump_0007: [1.0, 0.014000000432133675, 0.5016722679138184, 0.40133780241012573]
torus_bump_0008: [1.0, 0.01600000075995922, 0.5016722679138184, 0.40133780241012573]
torus_bump_0009: [1.0, 0.017999999225139618, 0.5016722679138184, 0.40133780241012573]
torus_bump_0010: [1.0, 0.019999999552965164, 0.5016722679138184, 0.40133780241

In [22]:
# Clean 2D front-view renders (matplotlib) from PLY meshes + optional MP4.
# Camera: elev=90, azim=0 (look straight into the hole), white bg, gold face, black edges.
#
# pip install trimesh imageio pillow torch matplotlib

import os, glob
import numpy as np
import imageio
import torch
import trimesh
import matplotlib
matplotlib.use("Agg")  # headless-safe
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


# ---------- CONFIG ----------
DATA_DIR = "torus_bump_1000_angle_classification"   # <- change if needed
OUT_DIR  = os.path.join(DATA_DIR, "renders_matplotlib")
MAKE_MP4 = True
OUT_MP4  = os.path.join(DATA_DIR, "dataset_front_view.mp4")
FPS      = 24

FIGSIZE  = (8, 6)   # inches
DPI      = 150
BG_COLOR = "white"
FACECLR  = "gold"   # same tone as your example
EDGECOLOR= "black"
EDGEW    = 0.2
ELEV     = 90
AZIM     = 0

os.makedirs(OUT_DIR, exist_ok=True)

# ---------- LOAD FILES / LABELS ----------
ply_files = sorted(glob.glob(os.path.join(DATA_DIR, "torus_bump_*.ply")))
if not ply_files:
    raise FileNotFoundError(f"No torus_bump_*.ply in {DATA_DIR}")

labels = torch.load(os.path.join(DATA_DIR, "labels.pt"), map_location="cpu", weights_only=False)

# ---------- FIXED AXIS LIMITS FROM FIRST MESH ----------
def read_mesh(pth):
    m = trimesh.load(pth, process=False)
    if not isinstance(m, trimesh.Trimesh):
        # if a Scene, merge
        m = trimesh.util.concatenate(m.dump())
    V = m.vertices.astype(np.float64)
    F = m.faces.astype(np.int32)
    return V, F

V0, F0 = read_mesh(ply_files[0])
mins, maxs = V0.min(axis=0), V0.max(axis=0)
center = (mins + maxs) / 2
extent = (maxs - mins).max() * 0.55  # padding
lims = np.stack([center - extent, center + extent], axis=0)  # (2,3)

# ---------- RENDER LOOP ----------
png_paths = []
for p in ply_files:
    name = os.path.splitext(os.path.basename(p))[0]
    arr = labels[name]
    print(f"{name}: {arr.tolist()}")  # print label

    V, F = read_mesh(p)
    tris = V[F]  # (T,3,3)

    fig = plt.figure(figsize=FIGSIZE, dpi=DPI, facecolor=BG_COLOR)
    ax = fig.add_subplot(111, projection="3d", facecolor=BG_COLOR)
    ax.set_axis_off()
    ax.set_box_aspect((1,1,1))

    ax.view_init(elev=ELEV, azim=AZIM)
    ax.set_xlim(lims[0,0], lims[1,0])
    ax.set_ylim(lims[0,1], lims[1,1])
    ax.set_zlim(lims[0,2], lims[1,2])

    coll = Poly3DCollection(tris, facecolor=FACECLR, edgecolor=EDGECOLOR, linewidths=EDGEW)
    ax.add_collection3d(coll)

    out_png = os.path.join(OUT_DIR, f"{name}.png")
    fig.savefig(out_png, facecolor=BG_COLOR, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    png_paths.append(out_png)

print(f"[OK] Saved {len(png_paths)} PNGs to {OUT_DIR}")

# ---------- OPTIONAL: MAKE MP4 FROM THE PNGs ----------
if MAKE_MP4:
    with imageio.get_writer(OUT_MP4, fps=FPS, codec="libx264", quality=8) as w:
        for png in png_paths:
            w.append_data(imageio.v3.imread(png))
    print(f"[OK] Wrote MP4: {OUT_MP4}")


torus_bump_0000: [1.0, 0.0, 0.5016722679138184, 0.40133780241012573]
torus_bump_0001: [1.0, 0.0020000000949949026, 0.5016722679138184, 0.40133780241012573]
torus_bump_0002: [1.0, 0.004000000189989805, 0.5016722679138184, 0.40133780241012573]
torus_bump_0003: [1.0, 0.006000000052154064, 0.5016722679138184, 0.40133780241012573]
torus_bump_0004: [1.0, 0.00800000037997961, 0.5016722679138184, 0.40133780241012573]
torus_bump_0005: [1.0, 0.009999999776482582, 0.5016722679138184, 0.40133780241012573]
torus_bump_0006: [1.0, 0.012000000104308128, 0.5016722679138184, 0.40133780241012573]
torus_bump_0007: [1.0, 0.014000000432133675, 0.5016722679138184, 0.40133780241012573]
torus_bump_0008: [1.0, 0.01600000075995922, 0.5016722679138184, 0.40133780241012573]
torus_bump_0009: [1.0, 0.017999999225139618, 0.5016722679138184, 0.40133780241012573]
torus_bump_0010: [1.0, 0.019999999552965164, 0.5016722679138184, 0.40133780241012573]
torus_bump_0011: [1.0, 0.02199999988079071, 0.5016722679138184, 0.401337

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (693, 693) to (704, 704) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


[OK] Saved 1000 PNGs to torus_bump_1000_angle_classification/renders_matplotlib
[OK] Wrote MP4: torus_bump_1000_angle_classification/dataset_front_view.mp4


In [292]:
scaler_t = MinMaxScaler()
scaler_t.fit(feature_range_thickness.reshape(-1, 1))
t = scaler_t.transform(np.array([0.1]).reshape(-1,1)).item()
t

0.5084745762711864

## Save template into file

In [5]:
radius=0.25
thickness=0.10
noise_scale=16
noise_strength=10
seed=50
bump_width=0.001
bump_height=0.001
bump_angle = 1

coords = np.linspace(-1, 1, 100)
x = np.stack(np.meshgrid(coords, coords, coords))
sdf = sdf_torus(x, radius, thickness)
verts, faces, normals, values = measure.marching_cubes(sdf, level=0)    

x_warp = gradient_noise(x, noise_scale, noise_strength) # no seed, random noise

angle = np.pi * bump_angle
gaussian_center = gaussian_center = np.array([np.cos(angle), np.sin(angle), 0]) * radius
x_dist = np.linalg.norm((x - gaussian_center[:, None, None, None]), axis=0)
x_bump = bump_height * np.e ** -(1. / bump_width * x_dist ** 2)
x_warp += -np.stack(np.gradient(x_bump))

x_warp = rearrange(x_warp, 'v h w d -> h w d v')
vertex_noise = interpn([np.arange(100) for _ in range(3)], x_warp, verts)
verts += vertex_noise

#scaler = MinMaxScaler(feature_range=(-0.5, 0.5))
#verts = scaler.fit_transform(verts)
print(verts.shape)
print(faces.shape)

igl.write_triangle_mesh(f"torus_bump_5000_two_scale_binary_bump_variable_noise_fixed_angle/template/template.ply", verts, faces)

(3496, 3)
(6992, 3)


True

## Save (min-max-scaled) labels into file

In [ ]:
scaler = MinMaxScaler()
labels = scaler.fit_transform(np.linspace(0, 1, 500).reshape(-1,1))
torch.save(labels.reshape(-1), "torus_bump_500_two/labels.pt")

In [459]:
labels = {}
feature_range = np.linspace(-1, 1, 500)
i=0
for x in tqdm(feature_range):
    filename = f'torus_bump_{i}'
    labels[filename] = x
    i = i+1

torch.save(labels, "C:/Users/Jakar/Downloads/Hippocampus_Study/torus_bump_500/torus_bump_500/labels.pt")
pd.DataFrame(list(labels.items()), columns=['shape', 'label']).to_csv("C:/Users/Jakar/Downloads/Hippocampus_Study/torus_bump_500/torus_bump_500/torus_bump_labels.csv")

100%|██████████| 500/500 [00:00<?, ?it/s]
